# proto-REINFORCE, part 3: Old Bessie's backward pass

On page 3 of Old Bessie, proto-REINFORCE learned to manage the van. Its forward pass goes from a table of bets
to the average $G_1$ of 1,000 runs; parts 1 and 2 build it. Here the forward pass is given, finished, in a folded
cell, and you build the backward pass: a bet picked at random, its slope, its update, and at the end the whole
loop. This notebook does not need parts 1 and 2.

**How it works.** Run the cells from top to bottom and fill in the 🎯 cells: in each of them, replace every `...`
with your code. Every other cell is ready to run. A folded cell with a title holds plumbing: run it, and open it
only if you are curious. The 💡 hints are folded cells too: double-click one to open it, and open the next one
only if you are still stuck. After each task, a check cell tells you whether your code works. The luck of the
game is in the kit: your code never deals with dice.

Start with the three folded cells below: Old Bessie's kit, the forward pass and the checks.

In [ ]:
#@title Old Bessie's kit (run me first) { display-mode: "form" }
# Old Bessie's kit. Run this cell first; you do not need to read it.
import random                   # Python's random numbers: random.randint(0, 3) gives 0, 1, 2 or 3
from math import exp            # e to the power: exp(0.4) is 1.4918...
import math as _kit_math

STATES = ['HEALTHY', 'WORN', 'SHAKY', 'FAILING']
ACTIONS = ['RUN', 'SERVICE', 'REPLACE']
HEALTHY, WORN, SHAKY, FAILING = 0, 1, 2, 3       # the rows of a table: the van's states
RUN, SERVICE, REPLACE = 0, 1, 2                  # the columns of a table: the actions
GAMMA = 0.9                # a week later, a dollar counts 0.9 times as much
WEEKS = 13                 # the weeks of a quarter: one run
RUNS = 1000                # the runs of every average, as on page 3

# The van of the arcade, as on page 1 (bessie.js): the week's money and the chances of what happens
_KIT_PROFIT = [95, 72, 40, 16]          # RUN: the route's money, by state
_KIT_BREAKDOWN = [0.02, 0.08, 0.28, 0.55]    # RUN: the chance of a breakdown, by state
_KIT_TOW = 280                       # a breakdown costs this on top, and the van is FAILING next week
_KIT_WEAR = [[0.35, 0.55, 0.1], [0.3, 0.55, 0.15], [0.45, 0.55, 0.0], [0.65, 0.35, 0.0]]     # RUN without a breakdown: wears 0, 1 or 2 levels
_KIT_SHOP = 50                       # SERVICE: a week in the garage
_KIT_REPAIR = [[1.0, 0.0, 0.0], [0.05, 0.7, 0.25], [0.45, 0.5, 0.05], [0.55, 0.42, 0.03]]     # SERVICE: repairs 0, 1 or 2 levels
_KIT_NEW_VAN = 130                   # REPLACE: a new van, HEALTHY next week
_KIT_START_BETS = [[0.4, -0.6, 0.1], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]]   # page 3's bets at random
_KIT_BEST_LINE = 244.03          # the best strategy's average G1 on the luck of the loop's last update
_KIT_FIRST_LUCK = 12                 # the luck of page 3's start: B.dice(12, 1000)
_KIT_LUCK_BASE = 2000               # update i of the loop plays the luck of seed 2000 + i, as page 3's movie
_KIT_PICKS = 2                       # the seed of Python's random for the loop's picks (play sets it)


def _kit_outcomes(state, action):
    """Every outcome of a week, in the order of the page: [running chance, next state, money]."""
    out = []
    if action == RUN:
        out.append([_KIT_BREAKDOWN[state], FAILING, _KIT_PROFIT[state] - _KIT_TOW])
        for d in range(3):
            p = (1 - _KIT_BREAKDOWN[state]) * _KIT_WEAR[state][d]
            if p > 0:
                out.append([p, min(FAILING, state + d), _KIT_PROFIT[state]])
    elif action == SERVICE:
        for d in range(3):
            p = _KIT_REPAIR[state][d]
            if p > 0:
                out.append([p, max(HEALTHY, state - d), -_KIT_SHOP])
    else:
        out.append([1, HEALTHY, -_KIT_NEW_VAN])
    running = 0
    for o in out:
        running = running + o[0]
        o[0] = running
    return out


_KIT_WEEK = []
for _s in range(4):
    _KIT_WEEK.append([])
    for _a in range(3):
        _KIT_WEEK[_s].append(_kit_outcomes(_s, _a))

# The luck: one stream of numbers between 0 and 1 (Mulberry32, the generator of the arcade and of page 3)
_KIT_LUCK = {"seed": _KIT_FIRST_LUCK, "state": _KIT_FIRST_LUCK, "update": 0}


def _kit_next():
    s = (_KIT_LUCK["state"] + 0x6D2B79F5) & 0xFFFFFFFF
    _KIT_LUCK["state"] = s
    t = ((s ^ (s >> 15)) * (1 | s)) & 0xFFFFFFFF
    t = ((t + (((t ^ (t >> 7)) * (61 | t)) & 0xFFFFFFFF)) & 0xFFFFFFFF) ^ t
    return ((t ^ (t >> 14)) & 0xFFFFFFFF) / 4294967296


def spin():
    """The ball of a roulette: a number between 0 and 1, every number as likely as any other."""
    return _kit_next()


def transition(state, action):
    """One week of Old Bessie, the transition dynamics of page 1. In this state, with this action, luck decides what
    happens, with the chances of page 1. Gives the van's state next week and the week's money:
        next_state, money = transition(HEALTHY, RUN)"""
    if state is Ellipsis or action is Ellipsis:
        print("⏳ transition waits for a state and an action: replace each ... with yours, then run the cell again.")
        return None, None
    try:
        outcomes = _KIT_WEEK[state][action]
    except (IndexError, TypeError):
        raise ValueError("transition needs a state from 0 to 3 and an action from 0 to 2; it got the state %r and "
                         "the action %r" % (state, action))
    u = _kit_next()
    for running, next_state, money in outcomes:
        if u < running:
            return next_state, money
    return outcomes[-1][1], outcomes[-1][2]


def same_luck():
    """The luck starts again from its beginning: the runs that follow meet the same weeks as the runs of every
    average before them, so two tables of bets are compared on the same luck."""
    _KIT_LUCK["state"] = _KIT_LUCK["seed"]


def new_luck():
    """New runs for the next update of proto-REINFORCE: the luck starts from a new seed, 2000 plus the number of the
    update."""
    _KIT_LUCK["update"] = _KIT_LUCK["update"] + 1
    _KIT_LUCK["seed"] = _KIT_LUCK_BASE + _KIT_LUCK["update"]
    _KIT_LUCK["state"] = _KIT_LUCK["seed"]


def _kit_set_luck(seed, update=0):
    """For the checks: the luck of this seed, from its beginning."""
    _KIT_LUCK["seed"] = seed
    _KIT_LUCK["state"] = seed
    _KIT_LUCK["update"] = update


def _kit_before_long():
    """play calls this before the loop: page 3's start, and the same picks for everyone."""
    _kit_set_luck(_KIT_FIRST_LUCK)
    random.seed(_KIT_PICKS)


def _kit_after_long():
    """play calls this after the loop: page 3's start again, for the cells above."""
    _kit_set_luck(_KIT_FIRST_LUCK)


def _kit_seed_picks(seed):
    """For the checks: Python's random from this seed."""
    random.seed(seed)


def _kit_save():
    """For the checks: the state of the luck and of Python's random."""
    return [dict(_KIT_LUCK), random.getstate()]


def _kit_restore(saved):
    """For the checks: the luck and Python's random as they were."""
    _KIT_LUCK.update(saved[0])
    random.setstate(saved[1])


def start_bets():
    """Page 3's table of bets picked at random, between -10 and 10: a fresh copy each time."""
    return copy_bets(_KIT_START_BETS)


def copy_bets(bets):
    """A copy of a table of bets: change the copy, and the table stays as it was."""
    copy = []
    for row in bets:
        copy.append(list(row))
    return copy


def _kit_is_number(x):
    return not isinstance(x, bool) and isinstance(x, (int, float)) and x == x


def _kit_is_table(table, columns=3):
    if not isinstance(table, list) or len(table) != len(STATES):
        return False
    for row in table:
        if not isinstance(row, list) or len(row) != columns:
            return False
        for x in row:
            if not _kit_is_number(x):
                return False
    return True


def _kit_short(value):
    text = repr(value).replace("Ellipsis", "...")
    return text if len(text) <= 60 else text[:57] + "..."


def _kit_print_table(table, places, mark):
    head = " " * 9
    for name in ACTIONS:
        head += "%10s" % name
    if mark:
        head += "    likeliest"
    print(head)
    for s in range(len(STATES)):
        line = "%-9s" % STATES[s]
        best = 0
        for a in range(len(ACTIONS)):
            line += "%10.*f" % (places, table[s][a])
            if table[s][a] > table[s][best]:
                best = a
        if mark:
            line += "    " + ACTIONS[best]
        print(line)


def show_bets(bets):
    """Print a table of bets: a row per state, a column per action, two decimals. Silent for None (a line above
    says why)."""
    if bets is None:
        return
    if not _kit_is_table(bets):
        print("⏳ No table of bets to show yet: show_bets needs four rows of three numbers. It got %s." % _kit_short(bets))
        return
    _kit_print_table(bets, 2, False)


def show_chances(table):
    """Print a probability table (a policy): a row per state, a column per action, two decimals, and the likeliest
    action of each row. Silent for None (a line above says why)."""
    if table is None:
        return
    if not _kit_is_table(table):
        print("⏳ No probability table to show yet: show_chances needs four rows of three numbers. It got %s."
              % _kit_short(table))
        return
    _kit_print_table(table, 2, True)


def show_counts(counts):
    """Print a table of counts: a row per state, a column per action, whole numbers."""
    if counts is None:
        return
    if not _kit_is_table(counts):
        print("⏳ No counts to show yet: show_counts needs four rows of three numbers. It got %s." % _kit_short(counts))
        return
    _kit_print_table(counts, 0, False)


def show_roulette(chances, ball=None):
    """Print the roulette of a row of chances: each action's slot, starting where the slot before it ends, on a wheel
    of 50 marks from 0 to 1. With a ball, a ^ marks where it is."""
    ok = isinstance(chances, list) and len(chances) == len(ACTIONS)
    if ok:
        for x in chances:
            if not _kit_is_number(x) or x < 0:
                ok = False
    if not ok:
        print("⏳ No roulette to show yet: show_roulette needs a row of three chances. It got %s." % _kit_short(chances))
        return
    start = 0
    for a in range(len(ACTIONS)):
        end = start + chances[a]
        left = int(round(start * 50))
        right = int(round(end * 50))
        print("%-8s %4.2f to %4.2f  |%s%s" % (ACTIONS[a], start, end, " " * left, "#" * (right - left)))
        start = end
    print(" " * 24 + "0" + " " * 49 + "1")
    if ball is not None and _kit_is_number(ball) and 0 <= ball < 1:
        print(" " * 24 + " " * int(ball * 50) + "^ the ball, %.2f" % ball)


def _kit_dollars(x):
    """$95, -$185, $119.30: whole dollars without cents."""
    if x == int(x):
        text = "$%d" % abs(int(x))
    else:
        text = "$%.2f" % abs(x)
    return "-" + text if x < 0 else text


def show_week(next_state, money):
    """Print one week of the van: its state next week and the week's money. Silent when there is no week yet (a
    line above says why)."""
    if next_state is None and money is None:
        return
    if isinstance(next_state, bool) or not isinstance(next_state, int) or not 0 <= next_state < len(STATES) or (
            not _kit_is_number(money)):
        print("⏳ No week to show yet: show_week needs a state and the week's money. It got %s and %s."
              % (_kit_short(next_state), _kit_short(money)))
        return
    print("Next week the van is %s. The week's money: %s." % (STATES[next_state], _kit_dollars(money)))


def show_money(label, money):
    """Print an amount of money, or a list of amounts, after a label: show_money("G1:", 154.4) prints G1: $154.40.
    A list longer than a run shows its first WEEKS. Silent when there is nothing to show yet (a line above says why)."""
    if money is None:
        return
    if isinstance(money, list):
        parts = []
        for k in range(len(money)):
            if k == WEEKS:
                parts.append("... (%s in all)" % format(len(money), ","))
                break
            if _kit_is_number(money[k]):
                parts.append(_kit_dollars(money[k]))
            else:
                parts.append(_kit_short(money[k]))
        print(label, ", ".join(parts) if parts else "none")
    elif _kit_is_number(money):
        print(label, _kit_dollars(money))
    else:
        print(label, _kit_short(money))


def show_value(label, value):
    """Print a value after a label: show_value("nabla =", 174). Silent for None (a line above says why)."""
    if value is None:
        return
    print(label, _kit_short(value))


def _kit_money(x):
    """-$172.24, $73.57"""
    text = "$%.2f" % abs(x)
    return "-" + text if x < 0 else text


def _kit_axis(x):
    """-$150, $250"""
    text = "$%d" % abs(int(round(x)))
    return "-" + text if x < 0 else text


def plot_rewards(points, best=_KIT_BEST_LINE, label="best strategy"):
    """The reward plot of page 3, as text: across, the number of updates; up, the average G1 after them. o is a
    point (one column for every few updates when there are many), and the row of dashes marks the best strategy's
    average G1 on the luck of the last update. Silent for None (a line above says why)."""
    if points is None:
        return
    ok = isinstance(points, list) and len(points) >= 2
    if ok:
        for x in points:
            if not _kit_is_number(x):
                ok = False
    if not ok:
        print("⏳ No reward plot yet: plot_rewards needs a list of at least two averages. It got %s." % _kit_short(points))
        return
    n = len(points) - 1                      # the updates
    columns = min(n + 1, 61)
    where = []                               # the update that each column shows
    for c in range(columns):
        where.append(int(round(c * n / (columns - 1))))
    values = []
    for i in where:
        values.append(points[i])
    low, high = min(values + [best]), max(values + [best])
    step = 10
    for size in (10, 25, 50, 100, 250, 500, 1000):
        step = size
        if (high - low) / size <= 18:
            break
    top = _kit_math.ceil(high / step) * step
    bottom = _kit_math.floor(low / step) * step
    rows = int(round((top - bottom) / step)) + 1
    grid = []
    for r in range(rows):
        grid.append([" "] * columns)
    best_row = int(round((top - best) / step))
    for c in range(columns):
        grid[best_row][c] = "-"
    for c in range(columns):
        r = int(round((top - values[c]) / step))
        grid[r][c] = "o"
    every = max(1, int(round(n / (columns - 1)))) if columns > 1 else 1
    print("The reward plot: the average G1 after each update" + (" (one o for every %d updates)" % every if every > 1
                                                                 else ""))
    print()
    for r in range(rows):
        line = "%8s |" % _kit_axis(top - r * step) + "".join(grid[r])
        if r == best_row:
            line += "  %s, %s" % (label, _kit_money(best))
        print(line.rstrip())
    print(" " * 9 + "+" + "-" * columns)
    ticks = 50 if n >= 200 else (10 if n >= 40 else 1)
    marks = [" "] * (columns + 6)
    for t in range(0, n + 1, ticks):
        c = int(round(t * (columns - 1) / n)) if n else 0
        text = str(t)
        for j in range(len(text)):
            if c + j < len(marks):
                marks[c + j] = text[j]
    print(" " * 10 + "".join(marks).rstrip() + "   updates")
    print("first point %s, last point %s" % (_kit_money(points[0]), _kit_money(points[-1])))


print("✅ Old Bessie's kit is ready.")

In [ ]:
#@title The forward pass of parts 1 and 2, given (run me too) { display-mode: "form" }
# The forward pass of parts 1 and 2, given: softmax, the roulette and their functions. You do not need to read this cell.


def softmax(row_of_bets):
    # A row of bets to a row of chances: e to the power of each bet, divided by their total, so that the chances
    # add up to 1 and a larger bet gets a larger chance. The largest bet is taken away first, as on page 3: the
    # chances stay the same, and exp never meets a number too large for it.
    largest = max(row_of_bets)
    powers = []
    for bet in row_of_bets:
        powers.append(exp(bet - largest))
    total = 0
    for power in powers:
        total = total + power
    chances = []
    for power in powers:
        chances.append(power / total)
    return chances


def probability_table(bets):
    # A table of bets to its probability table: softmax on every row.
    table = []
    for row in bets:
        table.append(softmax(row))
    return table



def make_roulette(chances):
    # chances: a row of chances, one per action.
    # Gives the ends of the slots: a running total of the chances, such as [0.6, 0.9, 1.0].
    ends = []
    running = 0                         # where the slots so far end: nowhere yet
    for chance in chances:
        running = running + chance      # this slot ends one chance further
        ends.append(running)            # remember where this slot ends
    return ends


def where_it_lands(ends, ball):
    # ends: the ends of the slots, from make_roulette. ball: a number between 0 and 1.
    # Gives the number of the first slot whose end is above the ball: 0 for RUN, 1 for SERVICE, 2 for REPLACE.
    for slot in range(len(ends)):
        if ball < ends[slot]:           # the ball is below the end of this slot
            return slot
    return len(ends) - 1                # a ball at the very end of the wheel: the last slot


def pick_action(policy, state):
    # policy: a probability table, one row per state. state: the van's state, the number of a row.
    # Gives an action picked at random with the chances of that row: 0 for RUN, 1 for SERVICE, 2 for REPLACE.
    chances = policy[state]             # the row of the state
    ends = make_roulette(chances)       # its roulette
    ball = spin()                       # the ball
    return where_it_lands(ends, ball)   # where the ball lands


def five_weeks(policy):
    # policy: a probability table. Plays five weeks of the van from HEALTHY, one after the other.
    # Gives the money of the five weeks, in a list.
    state = HEALTHY
    action = pick_action(policy, state)             # week 1: the policy picks an action in the van's state
    state, money_1 = transition(state, action)      # week 1: the van's week; state is now next week's state
    action = pick_action(policy, state)             # week 2, the policy's action
    state, money_2 = transition(state, action)      # week 2, the van's week
    action = pick_action(policy, state)             # week 3, the policy's action
    state, money_3 = transition(state, action)      # week 3, the van's week
    action = pick_action(policy, state)             # week 4, the policy's action
    state, money_4 = transition(state, action)      # week 4, the van's week
    action = pick_action(policy, state)             # week 5, the policy's action
    state, money_5 = transition(state, action)      # week 5, the van's week
    return [money_1, money_2, money_3, money_4, money_5]


def total_of_five(money):
    # money: the money of five weeks, in a list. Gives their total.
    total = money[0]                    # week 1
    total = total + money[1]            # week 2 added
    total = total + money[2]            # week 3 added
    total = total + money[3]            # week 4 added
    total = total + money[4]            # week 5 added
    return total


def play_run(policy, weeks):
    # policy: a probability table. weeks: how many weeks the run lasts.
    # Plays a run of the van from HEALTHY. Gives its rewards: the money of every week, in a list.
    rewards = []
    state = HEALTHY
    for week in range(weeks):
        action = pick_action(policy, state)         # the policy picks an action in the van's state
        state, money = transition(state, action)    # the van's week
        rewards.append(money)                       # remember the week's money
    return rewards


def cumulative_reward(rewards):
    # rewards: the money of every week of a run, in a list.
    # Gives the run's cumulative reward G1: week 1's money counts in full, and each later week's money counts
    # GAMMA = 0.9 times as much as the week before.
    G1 = 0
    weight = 1                          # week 1 counts in full
    for money in rewards:
        G1 = G1 + weight * money        # add the week's money, times its weight
        weight = weight * GAMMA         # the next week counts GAMMA times as much
    return G1


def play_runs(policy, weeks, how_many):
    # Plays how_many runs of weeks weeks with the same policy.
    # Gives the runs in a list, each run the list of its rewards.
    runs = []
    for k in range(how_many):
        run = play_run(policy, weeks)   # one run
        runs.append(run)                # remember the run
    return runs


def cumulative_rewards(runs):
    # runs: a list of runs, each the list of its rewards.
    # Gives the cumulative reward G1 of every run, in a list.
    G1s = []
    for run in runs:
        G1s.append(cumulative_reward(run))      # the run's G1
    return G1s


def average(numbers):
    # numbers: a list of numbers. Gives their average: their total divided by how many there are.
    total = 0
    for x in numbers:
        total = total + x               # add x to the total
    return total / len(numbers)         # the total divided by how many numbers there are


def average_G1(bets):
    # bets: a table of bets. Gives the average G1 of its policy over RUNS runs of WEEKS weeks, in cents.
    same_luck()                         # every average plays the same luck: the same runs for every table of bets
    policy = probability_table(bets)            # the probability table of the bets
    runs = play_runs(policy, WEEKS, RUNS)       # RUNS runs of WEEKS weeks with the policy
    G1s = cumulative_rewards(runs)              # the G1 of every run
    return round(average(G1s), 2)       # their average, in cents as on page 3


print("✅ The forward pass of parts 1 and 2 is ready: probability_table, play_run, play_runs, cumulative_rewards, average_G1.")

In [ ]:
#@title The checks of part 3 (run me too) { display-mode: "form" }
# The checks of notebook 1 and play. You do not need to read this cell.
# A check prints one short message and never stops the notebook.

_N1_TESTS = {'pick_bet': [{'about': 'pick 1 of the check', 'args': [], 'kind': 'numbers', 'want': [1, 2], 'tol': 0, 'setup': [['_kit_seed_picks', [1]]]}, {'about': 'pick 2 of the check', 'args': [], 'kind': 'numbers', 'want': [0, 0], 'tol': 0, 'setup': [['_kit_seed_picks', [2]]]}, {'about': 'pick 3 of the check', 'args': [], 'kind': 'numbers', 'want': [2, 2], 'tol': 0, 'setup': [['_kit_seed_picks', [5]]]}, {'about': 'pick 4 of the check', 'args': [], 'kind': 'numbers', 'want': [1, 1], 'tol': 0, 'setup': [['_kit_seed_picks', [8]]]}, {'about': 'pick 5 of the check', 'args': [], 'kind': 'numbers', 'want': [2, 1], 'tol': 0, 'setup': [['_kit_seed_picks', [13]]]}, {'about': 'pick 6 of the check', 'args': [], 'kind': 'numbers', 'want': [1, 1], 'tol': 0, 'setup': [['_kit_seed_picks', [21]]]}], 'slope': [{'about': "page 3's bets, HEALTHY and RUN, on page 3's luck", 'args': [[[0.4, -0.6, 0.1], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]], 0, 0, 0.01], 'kind': 'number', 'want': 174, 'tol': 0, 'keep': [0], 'setup': [['_kit_set_luck', [12]]]}, {'about': "page 3's bets, WORN and SERVICE, on page 3's luck", 'args': [[[0.4, -0.6, 0.1], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]], 1, 1, 0.01], 'kind': 'number', 'want': 6, 'tol': 0, 'keep': [0], 'setup': [['_kit_set_luck', [12]]]}, {'about': "page 3's bets, FAILING and SERVICE, on page 3's luck", 'args': [[[0.4, -0.6, 0.1], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]], 3, 1, 0.01], 'kind': 'number', 'want': -3, 'tol': 0, 'keep': [0], 'setup': [['_kit_set_luck', [12]]]}, {'about': "page 3's bets, HEALTHY and RUN, on luck 2 of the check", 'args': [[[0.4, -0.6, 0.1], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]], 0, 0, 0.01], 'kind': 'number', 'want': 224, 'tol': 0, 'keep': [0], 'setup': [['_kit_set_luck', [2001]]]}], 'update_bet': [{'about': "page 3's update 1, HEALTHY and RUN with nabla 174 and alpha 0.01", 'args': [[[0.4, -0.6, 0.1], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]], 0, 0, 174, 0.01], 'kind': 'number', 'want': 2.14, 'tol': 1e-06, 'table': {'arg': 0, 'want': [[2.14, -0.6, 0.1], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]]}}, {'about': "page 3's update 2, HEALTHY and REPLACE with nabla -13 and alpha 0.01", 'args': [[[2.14, -0.6, 0.1], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]], 0, 2, -13, 0.01], 'kind': 'number', 'want': -0.03, 'tol': 1e-06, 'table': {'arg': 0, 'want': [[2.14, -0.6, -0.03], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]]}}, {'about': "page 3's bets, SHAKY and REPLACE with nabla 0 and alpha 0.01", 'args': [[[0.4, -0.6, 0.1], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]], 2, 2, 0, 0.01], 'kind': 'number', 'want': 10.0, 'tol': 1e-06, 'table': {'arg': 0, 'want': [[0.4, -0.6, 0.1], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10.0], [0.4, 0.6, -0.4]]}}, {'about': "page 3's bets, WORN and SERVICE with nabla 6 and alpha 0.02", 'args': [[[0.4, -0.6, 0.1], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]], 1, 1, 6, 0.02], 'kind': 'number', 'want': -0.68, 'tol': 1e-06, 'table': {'arg': 0, 'want': [[0.4, -0.6, 0.1], [-0.7, -0.68, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]]}}], 'proto_reinforce': [{'about': "page 3's bets, 3 updates with hop 0.01 and alpha 0.01, on the luck and the picks of the loop cell", 'args': [[[0.4, -0.6, 0.1], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]], 3, 0.01, 0.01], 'kind': 'numbers', 'want': [-172.24, 123.82, 127.4, 119.29], 'tol': 0.010000001, 'table': {'arg': 0, 'want': [[2.64, -0.54, 0.1], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]]}, 'setup': [['_kit_before_long', []]]}, {'about': "page 3's bets, 2 updates with hop 0.01 and alpha 0.01, on the luck and other picks", 'args': [[[0.4, -0.6, 0.1], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]], 2, 0.01, 0.01], 'kind': 'numbers', 'want': [-172.24, -146.38, -155.82], 'tol': 0.010000001, 'table': {'arg': 0, 'want': [[0.4, -1.06, 0.1], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.14]]}, 'setup': [['_kit_before_long', []], ['_kit_seed_picks', [10]]]}, {'about': "page 3's bets, 2 updates with hop 0.01 and alpha 0.02, on the luck and the picks of the loop cell", 'args': [[[0.4, -0.6, 0.1], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]], 2, 0.01, 0.02], 'kind': 'numbers', 'want': [-172.24, 169.88, 169.23], 'tol': 0.010000001, 'table': {'arg': 0, 'want': [[4.88, -0.6, 0.1], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]]}, 'setup': [['_kit_before_long', []]]}]}
_N1_SLIPS = {'pick_bet': [{'answers': [(0, 2), (0, 0), (2, 1), (0, 1), (1, 1), (0, 1)], 'hint': 'a row is one of the 4 states: random.randint(0, 3) gives 0, 1, 2 or 3'}, {'answers': [(2, 2), (1, 0), (3, 2), (2, 1), (3, 1), (2, 1)], 'hint': 'the rows count from 0: random.randint(0, 3) gives 0, 1, 2 or 3'}, {'answers': [(1, 0), (0, 0), (2, 2), (1, 2), (2, 2), (1, 3)], 'hint': 'a column is one of the 3 actions: random.randint(0, 2) gives 0, 1 or 2'}], 'slope': [{'answers': [-9244, -364, -498, -8874], 'hint': 'raise the bet by the hop: hopped[state][action] + hop'}, {'answers': [0, 0, 0, 0], 'hint': 'After is the average G1 of the hopped bets: average_G1(hopped)'}, {'answers': [2, 0, 0, 2], 'hint': 'divide the change by the hop: (after - before) / hop'}, {'answers': [-174, -6, 3, -224], 'hint': 'the change is after - before: when After is higher, the slope is positive'}], 'update_bet': [{'answers': [-1.34, 0.23, 10.0, -0.92], 'hint': 'the average G1 has to go up, so the update adds: new_bet = w + alpha * nabla'}, {'answers': [174.4, -12.9, 10, 5.2], 'hint': 'alpha keeps the move small: new_bet = w + alpha * nabla'}, {'answers': [1.74, -0.13, 0.0, 0.12], 'hint': 'the new bet starts from the bet as it is: new_bet = w + alpha * nabla'}], 'proto_reinforce': [{'answers': [[-172.24, 224, 6, 0], [-172.24, -46, 26], [-172.24, 224, 0]], 'hint': 'a point is the average G1 after the update: points.append(average_G1(bets))'}, {'answers': [[-172.24, 123.82, 146.93, 157.62], [-172.24, 123.82, 146.93], [-172.24, 169.88, 169.7]], 'hint': 'each update picks its bet at random: state, action = pick_bet()'}, {'answers': [[-172.24, -156.92, -165.41, -155.91], [-172.24, -156.92, -165.41], [-172.24, -156.92, -165.41]], 'hint': 'each update moves its bet with your update_bet: update_bet(bets, state, action, nabla, alpha)'}, {'answers': [[-172.24, 119.29], [-172.24, -155.82], [-172.24, 169.23]], 'hint': 'one point after every update: points.append(average_G1(bets)) goes inside the loop'}, {'answers': [[-172.24, 123.82, 127.4, 119.29], [-172.24, -146.38, -155.82], [-172.24, 123.82, 127.4]], 'hint': 'the update moves the bet by alpha times nabla: update_bet(bets, state, action, nabla, alpha)'}]}
_N1_SUCCESS = {'pick_bet': "pick_bet works: a row from 0 to 3 and a column from 0 to 2, as Python's random picked them.", 'slope': "slope works: page 3's update 1, HEALTHY and RUN, gives nabla = 174, as on page 3.", 'update_bet': "update_bet works: page 3's update 1 moves the bet of HEALTHY and RUN from 0.40 to 2.14.", 'proto_reinforce': 'proto_reinforce works: its first 3 updates give the points -$172.24, $123.82, $127.40, $119.29.'}
_N1_DEFAULT = {'pick_bet': 'state = random.randint(0, 3), then action = random.randint(0, 2)', 'slope': 'hopped[state][action] + hop, then after = average_G1(hopped), then nabla = (after - before) / hop', 'update_bet': 'new_bet = w + alpha * nabla', 'proto_reinforce': 'state, action = pick_bet(), then nabla = slope(bets, state, action, hop), then update_bet(bets, state, action, nabla, alpha), then points.append(average_G1(bets))'}
_N1_KEEP_HINT = {'pick_bet': 'pick_bet takes no inputs', 'slope': 'slope leaves the table of bets as it is: the hop goes into the copy, hopped', 'update_bet': 'update_bet changes only the bet of state and action: bets[state][action] = round(new_bet, 2)', 'proto_reinforce': 'proto_reinforce changes the table of bets it gets: update_bet(bets, state, action, nabla, alpha) on that table, with no copy'}
_N1_TASKS = {'pick_bet': 'task A', 'slope': 'task B', 'update_bet': 'task C', 'proto_reinforce': 'task D'}
_N1_NEEDS = {'pick_bet': [], 'slope': ['average_G1'], 'update_bet': [], 'proto_reinforce': ['pick_bet', 'slope', 'update_bet', 'average_G1']}
_N1_LONG = ['proto_reinforce']
_N1_NONE_HINT = {}
_N1_DICE_OF = 'spin'
_N1_GAPS = {}
_N1_WORKING = {}
_N1_DICE = {}


def _n1_num(x):
    """A number with at most 4 decimals: 154.3961, 2.14, 174."""
    if isinstance(x, bool) or not isinstance(x, (int, float)):
        return repr(x)
    if x != x or x in (float("inf"), float("-inf")):
        return repr(x)
    if isinstance(x, int):
        return str(x)
    text = ("%.4f" % x).rstrip("0").rstrip(".")
    if text == "-0":
        text = "0"
    if text == "0" and x != 0:
        text = "%.2g" % x
    return text


def _n1_text(value):
    """Numbers with at most 4 decimals, lists of them in brackets."""
    if isinstance(value, (list, tuple)):
        parts = []
        for x in value:
            parts.append(_n1_text(x))
        return "[" + ", ".join(parts) + "]"
    return _n1_num(value)


def _n1_short(value):
    """A value as text, cut short when it is long."""
    try:
        text = _n1_text(value)
    except Exception:
        text = "a value that cannot be printed"
    if len(text) > 72:
        text = text[:69] + "..."
    return text


def _n1_is_number(x):
    return not isinstance(x, bool) and isinstance(x, (int, float)) and x == x


def _n1_same(kind, got, want, tol):
    """True when got is the right answer want, within tol."""
    try:
        if kind == "number":
            return _n1_is_number(got) and abs(got - want) <= tol
        if kind == "action":
            return _n1_is_number(got) and got == want
        if kind in ("numbers", "table"):
            if not isinstance(got, (list, tuple)) or len(got) != len(want):
                return False
            for k in range(len(want)):
                if not _n1_same("numbers" if kind == "table" else "number", got[k], want[k], tol):
                    return False
            return True
    except Exception:
        return False
    return False


def _n1_like(got, answer, tol):
    """True when got has the shape and the numbers of a slip's answer, within tol: a slip may give a number where a
    list is due (return total), or a list of numbers where a list of lists is due."""
    if isinstance(answer, (list, tuple)):
        if not isinstance(got, (list, tuple)) or len(got) != len(answer):
            return False
        for k in range(len(answer)):
            if not _n1_like(got[k], answer[k], tol):
                return False
        return True
    return _n1_is_number(got) and _n1_is_number(answer) and abs(got - answer) <= tol


def _n1_copy(value):
    """A deep copy of lists of lists of numbers."""
    if isinstance(value, list):
        out = []
        for x in value:
            out.append(_n1_copy(x))
        return out
    return value


def _n1_dice(seed, first, count):
    """The dice of count runs of a seed from run first, rolled by the kit once per seed and kept."""
    rolled = _N1_DICE.get(seed)
    if rolled is None or len(rolled) < first + count:
        if len(_N1_DICE) > 3:
            _N1_DICE.clear()
        rolled = globals()[_N1_DICE_OF](seed, max(first + count, 10))
        _N1_DICE[seed] = rolled
    return rolled[first:first + count]


def _n1_args(test):
    """Fresh inputs for a test: copies of the lists, and the dice it names."""
    args = []
    for a in test["args"]:
        if isinstance(a, dict) and "dice" in a:
            args.append(_n1_dice(a["dice"][0], a["dice"][1], a["dice"][2]))
        elif isinstance(a, dict) and "run" in a:
            args.append(_n1_dice(a["run"][0], a["run"][1], 1)[0])
        else:
            args.append(_n1_copy(a))
    return args


def _n1_has_gap(fn):
    """True when the code of fn still holds a ... to fill in (comments do not count)."""
    code = getattr(fn, "__code__", None)
    if code is None:
        return False
    if code in _N1_GAPS:
        return _N1_GAPS[code]
    # A gap such as "total = ..." stays in the compiled function; a gap such as "if ...:" only in the source, which
    # Jupyter and Colab keep for every cell
    gap = Ellipsis in code.co_consts
    if not gap:
        import inspect
        try:
            for line in inspect.getsource(fn).splitlines():
                if "..." in line.split("#")[0]:
                    gap = True
        except Exception:
            pass
    _N1_GAPS[code] = gap
    return gap


def _n1_holds_gap(value):
    """True when a value, or a list inside it, still holds a ... (a gap of a cell above)."""
    if value is Ellipsis:
        return True
    if isinstance(value, (list, tuple)):
        for x in value:
            if _n1_holds_gap(x):
                return True
    return False


def _n1_uses(fn, other):
    """True when the code of fn calls the function named other."""
    code = getattr(fn, "__code__", None)
    return code is not None and other in code.co_names


def _n1_key(name, seen=()):
    """The code of the function called name and of the functions it needs: a new version of any of them is checked
    again."""
    if name in seen:
        return ()
    fn = globals().get(name)
    key = [getattr(fn, "__code__", None)]
    for other in _N1_NEEDS.get(name, []):
        key.extend(_n1_key(other, seen + (name,)))
    return tuple(key)


def _n1_call(fn, args):
    """Run fn with its prints silenced: {"answer", "error" (None when the call went through), "line" (the line of
    your code where it happened)}. Never raises."""
    import contextlib, io, traceback
    run = {"answer": None, "error": None, "line": None}
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            run["answer"] = fn(*args)
    except Exception as error:
        run["error"] = "%s: %s" % (type(error).__name__, error)
        try:
            skip = (_n1_call.__code__.co_filename, globals()[_N1_DICE_OF].__code__.co_filename)
            for frame in traceback.extract_tb(error.__traceback__):
                if frame.filename not in skip and frame.line:
                    run["line"] = frame.line.strip()
        except Exception:
            run["line"] = None
    return run


def _n1_explain(run):
    """A hint for an error."""
    text = run["error"]
    if "NoneType" in text:
        return 'a value in that line is None: a function it calls gave nothing back (is that function finished, with its return line?), or a name holds None'
    if "ellipsis" in text:
        return "a ... is still in a function that this one calls: finish that function first"
    if "list indices must be integers" in text:
        return 'a place in [ ] must be a whole number: a state or an action, such as policy[state] for the row of a state and row[action] for one of its numbers'
    if "is not subscriptable" in text:
        return "that line uses [ ] on a single number: only a list has places to look up"
    if "is not iterable" in text:
        return ("a single number stands where a list was expected: check the names on the left of each = (update_bet "
                "changes the table itself and gives back only the new bet)")
    if 'transition needs' in text:
        return 'the van got a state or an action it does not know: check what pick_action gives back'
    if "ZeroDivisionError" in text:
        return "a division by zero: divide by a number that cannot be 0, such as the total or len(...)"
    if "NameError" in text:
        return "a name in that line is not defined: check its spelling, and run the cells above it first"
    if "can only concatenate" in text or "unsupported operand" in text:
        return "that line mixes a list and a number: take one number of the list with [ ] first"
    if "positional argument" in text:
        return "a function got more or fewer inputs than it takes: compare the call with the function's def line"
    if "out of range" in text:
        return "a place in [ ] is past the end of the list: count from 0, the last place is len(...) - 1"
    return "read that line again, compare it with the hints, run the function's cell, then this check again"


def _n1_failure(name, fn):
    """None when fn passes every test of its task, else (head, what, hint). The kit's state (its luck) is put back
    afterwards when the kit can save it."""
    save = globals().get("_kit_save")
    saved = save() if callable(save) else None
    try:
        return _n1_failure_of(name, fn)
    finally:
        if saved is not None:
            globals()["_kit_restore"](saved)


def _n1_failure_of(name, fn):
    tests = _N1_TESTS[name]
    for k in range(len(tests)):
        test = tests[k]
        args = _n1_args(test)
        kept = _n1_copy(args)
        for setup in test.get("setup", []):
            globals()[setup[0]](*setup[1])
        run = _n1_call(fn, args)
        head = "On " + test["about"]
        if run["error"] is not None:
            where = ""
            if run["line"]:
                where = ", in the line  %s " % run["line"]
            return (head, "%s stopped with an error, %s%s" % (name, run["error"], where), _n1_explain(run))
        got = run["answer"]
        if not _n1_same(test["kind"], got, test["want"], test["tol"]):
            if got is None:
                return (head, "%s gave None" % name, _N1_NONE_HINT.get(
                    name, "a function gives its answer with return: keep the return line at its end"))
            hint = _N1_DEFAULT[name]
            for slip in _N1_SLIPS[name]:
                answer = slip["answers"][k]
                if answer is not None and _n1_like(got, answer, test["tol"]):
                    hint = slip["hint"]
                    break
            return (head, "%s should give %s, and yours gives %s" % (name, _n1_short(test["want"]),
                                                                    _n1_short(got)), hint)
        for i in test.get("keep", []):
            if args[i] != kept[i]:
                return (head, "%s changed its input %s" % (name, _n1_short(kept[i])), _N1_KEEP_HINT[name])
        if test.get("table"):
            t = test["table"]
            if not _n1_same("table", args[t["arg"]], t["want"], test["tol"]):
                return (head, "after %s the table of bets should be %s, and it is %s" % (
                    name, _n1_short(t["want"]), _n1_short(args[t["arg"]])), _N1_KEEP_HINT[name])
    return None


def _n1_works(name):
    """True when the function called name exists, is finished, the functions it uses work, and it passes its tests.
    Remembered for each version of it."""
    fn = globals().get(name)
    if not callable(fn) or getattr(fn, "__code__", None) is None or _n1_has_gap(fn):
        return False
    for other in _N1_NEEDS.get(name, []):
        if _n1_uses(fn, other) and not _n1_works(other):
            return False
    if name not in _N1_TESTS:
        return True
    key = _n1_key(name)
    if key not in _N1_WORKING:
        _N1_WORKING[key] = _n1_failure(name, fn) is None
    return _N1_WORKING[key]


def _n1_waits(name, fn, cell):
    """One calm line and True when fn cannot run yet: it is not finished, or a function it uses does not work yet."""
    task = ""
    if name in _N1_TASKS:
        task = " (%s)" % _N1_TASKS[name]
    if _n1_has_gap(fn):
        print("⏳ %s is not finished yet%s: replace each ... with your code, run its cell, then %s again."
              % (name, task, cell))
        return True
    for other in _N1_NEEDS.get(name, []):
        if not _n1_uses(fn, other) or _n1_works(other):
            continue
        if other in _N1_TASKS:
            needed = globals().get(other)
            what = "does not work yet"
            if callable(needed) and _n1_has_gap(needed):
                what = "is not finished yet"
            print("⏳ %s uses your %s (%s), which %s. Get check_%s(%s) to ✅ first, then run %s again."
                  % (name, other, _N1_TASKS[other], what, other, other, cell))
            return True
        # a given function: it waits for the tasks it uses
        for deeper in _N1_NEEDS.get(other, []):
            if deeper in _N1_TASKS and not _n1_works(deeper):
                print("⏳ %s uses %s, which uses your %s (%s). Get check_%s(%s) to ✅ first, then run %s again."
                      % (name, other, deeper, _N1_TASKS[deeper], deeper, deeper, cell))
                return True
        print("⏳ %s uses %s, which does not work yet: run its cell again, then %s." % (name, other, cell))
        return True
    return False


def _n1_check(name, fn):
    """Print one message about fn: it works, it is not finished yet, or the first test it fails and a hint."""
    if not callable(fn):
        print("❌ check_%s needs your function itself: check_%s(%s), with no brackets after %s." % (
            name, name, name, name))
        return
    if _n1_waits(name, fn, "this check"):
        return
    failure = _n1_failure(name, fn)
    if failure is None:
        print("✅ " + _N1_SUCCESS[name])
        return
    head, what, hint = failure
    print("❌ %s: %s." % (head, what))
    print("   💡 Hint: " + hint + ".")


def _n1_safe_check(name, fn):
    try:
        _n1_check(name, fn)
    except Exception as error:
        print("❌ The check could not run (%s). Run the folded cells at the top again, then this cell." % error)


def check_pick_bet(fn):
    """Task a: the bet of several picks, each from a seed of Python's random that the check sets"""
    _n1_safe_check("pick_bet", fn)

def check_slope(fn):
    """Task b: the slope of several bets, each on luck the check deals"""
    _n1_safe_check("slope", fn)

def check_update_bet(fn):
    """Task c: the new bet of several updates, and the table of bets after each"""
    _n1_safe_check("update_bet", fn)

def check_proto_reinforce(fn):
    """Task d: the points and the table of bets after a few updates, on the luck and the picks the check sets"""
    _n1_safe_check("proto_reinforce", fn)


def play(fn, *inputs):
    """Runs one of your functions and gives its answer: play(average_G1, bets) is
    average_G1(bets). When the function is not finished yet, uses a function that does not work yet, gets
    None from a cell above or stops with an error, play prints one calm line and gives None, so the cells below
    still run."""
    import time
    try:
        if not callable(fn):
            print("⏳ play needs your function itself, as in play(average_G1, bets).")
            return None
        name = getattr(fn, "__name__", "your function")
        if _n1_waits(name, fn, "this cell"):
            return None
        for x in inputs:
            if x is None:
                print("⏳ %s waits for a value that a cell above did not give yet (its ⏳ line says why)." % name)
                return None
            if _n1_holds_gap(x):
                print("⏳ %s waits for a value that still holds a ...: fill it in above, run its cell, then this cell."
                      % name)
                return None
        if name in _N1_LONG and not _n1_works(name):
            print("⏳ %s waits until it passes its check: get check_%s(%s) to ✅ first, then run this cell again."
                  % (name, name, name))
            return None
        long = name in _N1_LONG
        if long and callable(globals().get("_kit_before_long")):
            globals()["_kit_before_long"]()
        start = time.time()
        try:
            answer = fn(*inputs)
        except Exception as error:
            print("❌ %s stopped with an error, %s: %s. Its check says more." % (name, type(error).__name__, error))
            return None
        finally:
            if long and callable(globals().get("_kit_after_long")):
                globals()["_kit_after_long"]()
        seconds = time.time() - start
        if seconds > 5:
            print("⏱️ %s took %d seconds." % (name, round(seconds)))
        return answer
    except Exception as error:
        print("❌ play could not run (%s). Run the folded cells at the top again, then this cell." % error)
        return None

_N1_TABLE = [[0.6, 0.3, 0.1], [0.5, 0.4, 0.1], [0.2, 0.5, 0.3], [0.1, 0.3, 0.6]]
_N1_START_BETS = [[0.4, -0.6, 0.1], [-0.7, -0.8, -6.9], [-0.4, 0.1, 10], [0.4, 0.6, -0.4]]


def _n1_ready(name, cell):
    """True when your function called name exists and passes its check; else one calm line."""
    fn = globals().get(name)
    if not callable(fn):
        print("⏳ %s is not there yet: run the cell of %s first, then %s." % (name, _N1_TASKS[name], cell))
        return False
    if _n1_waits(name, fn, cell):
        return False
    if not _n1_works(name):
        print("⏳ This cell waits until %s passes its check: get check_%s(%s) to ✅ first, then run %s again."
              % (name, name, name, cell))
        return False
    return True


def _n1_chances_ready(chances, cell):
    """True when chances is a row of three numbers; else one calm line."""
    if chances is None or _n1_holds_gap(chances):
        print("⏳ %s waits for a row of chances: finish my_table first (warm-up 1), then run it again." % cell)
        return False
    if not isinstance(chances, list) or len(chances) != 3:
        print("⏳ %s needs a row of three chances, such as my_table[HEALTHY]." % cell)
        return False
    return True


def _n1_policy_ready(policy, cell):
    """True when policy is a probability table without gaps; else one calm line."""
    if policy is None or _n1_holds_gap(policy):
        print("⏳ %s waits for a probability table: finish my_table first (warm-up 1), then run it again." % cell)
        return False
    return True


def check_table(table):
    """Warm-up 1: my_table, the made-up probability table."""
    try:
        if _n1_holds_gap(table):
            print("⏳ my_table is not finished yet (warm-up 1): replace each ... with a row of three chances, run its "
                  "cell, then this check again.")
            return
        if not isinstance(table, list) or len(table) != 4:
            print("❌ my_table should be a list of four rows, one per state, and yours is %s." % _n1_short(table))
            print("   💡 Hint: write one row per line, each in [ ] and followed by a comma, all inside the outer [ ].")
            return
        for s in range(4):
            row = table[s]
            if not _n1_same("numbers", row, _N1_TABLE[s], 1e-9):
                print("❌ %s's row should be %s, and yours is %s." % (STATES[s], _n1_text(_N1_TABLE[s]),
                                                                   _n1_short(row)))
                hint = "copy the row from the table above, three numbers in the order RUN, SERVICE, REPLACE"
                for other in range(4):
                    if other != s and _n1_same("numbers", row, _N1_TABLE[other], 1e-9):
                        hint = ("that is %s's row: the rows go in the order HEALTHY, WORN, SHAKY, FAILING"
                                % STATES[other])
                if isinstance(row, list) and len(row) == 3 and _n1_same("numbers", sorted(row), sorted(_N1_TABLE[s]),
                                                                        1e-9) and hint.startswith("copy"):
                    hint = "the numbers of a row go in the order of the columns: RUN, SERVICE, REPLACE"
                print("   💡 Hint: " + hint + ".")
                return
        print("✅ my_table is the made-up probability table: four rows, one per state, and each row adds up to 1.")
    except Exception as error:
        print("❌ The check could not run (%s). Run the folded cells at the top again, then this cell." % error)


def check_reading(chance, row):
    """Warm-up 2: a number and a row read from my_table."""
    try:
        if chance is Ellipsis or _n1_holds_gap(row):
            print("⏳ Warm-up 2 is not finished yet: replace each ... with your code, run its cell, then this check "
                  "again.")
            return
        want_chance = _N1_TABLE[FAILING][REPLACE]
        want_row = _N1_TABLE[SHAKY]
        if not _n1_same("number", chance, want_chance, 1e-9):
            print("❌ chance should be %s, the chance of REPLACE when the van is FAILING, and yours is %s."
                  % (_n1_text(want_chance), _n1_short(chance)))
            hint = "the row comes first, then the column: my_table[FAILING][REPLACE]"
            if _n1_same("number", chance, _N1_TABLE[FAILING][SERVICE], 1e-9) or _n1_same(
                    "number", chance, _N1_TABLE[FAILING][RUN], 1e-9):
                hint = "that is another column of FAILING's row: REPLACE is the column REPLACE, number 2"
            print("   💡 Hint: " + hint + ".")
            return
        if not _n1_same("numbers", row, want_row, 1e-9):
            print("❌ row should be SHAKY's row, %s, and yours is %s." % (_n1_text(want_row), _n1_short(row)))
            print("   💡 Hint: a row is the table with one [ ]: my_table[SHAKY].")
            return
        print("✅ Both right: my_table[FAILING][REPLACE] is %s, and my_table[SHAKY] is %s." % (
            _n1_text(want_chance), _n1_text(want_row)))
    except Exception as error:
        print("❌ The check could not run (%s). Run the folded cells at the top again, then this cell." % error)


def check_garage(next_state, money):
    """Warm-up 3: one week of a WORN van in the garage, transition(WORN, SERVICE)."""
    try:
        if next_state is None and money is None:
            print("⏳ Warm-up 3 is not finished yet: replace each ... with your code, run its cell, then this check "
                  "again.")
            return
        for running, state, cash in _KIT_WEEK[WORN][SERVICE]:
            if next_state == state and money == cash:
                print("✅ A week of a WORN van in the garage: next week it is %s, and the week's money is %s." % (
                    STATES[next_state], _kit_dollars(money)))
                return
        print("❌ A WORN van in the garage costs %s for the week and comes back HEALTHY or WORN; your week gave the "
              "state %s and the money %s." % (_kit_dollars(_KIT_SHOP), _n1_short(next_state), _n1_short(money)))
        hint = "call transition with the state WORN first, then the action SERVICE: transition(WORN, SERVICE)"
        if money == -_KIT_NEW_VAN:
            hint = "that is the money of REPLACE: the action of the garage is SERVICE"
        elif _n1_is_number(money) and money != -_KIT_SHOP:
            hint = "that money comes from the route, the action RUN: the action of the garage is SERVICE"
        print("   💡 Hint: " + hint + ".")
    except Exception as error:
        print("❌ The check could not run (%s). Run the folded cells at the top again, then this cell." % error)


def _n1_softmax(row):
    largest = max(row)
    powers = []
    for x in row:
        powers.append(exp(x - largest))
    total = 0
    for x in powers:
        total = total + x
    out = []
    for x in powers:
        out.append(x / total)
    return out


def check_my_bets(bets):
    """Warm-up 4: page 3's bets with the bet of HEALTHY and REPLACE set to 5."""
    try:
        if bets is None or _n1_holds_gap(bets):
            print("⏳ Warm-up 4 is not finished yet: replace the ... with your code, run its cell, then this check "
                  "again.")
            return
        if not _kit_is_table(bets):
            print("❌ my_bets should be a table of bets, four rows of three numbers, and it is %s." % _n1_short(bets))
            print("   💡 Hint: start from my_bets = start_bets(), then change one bet.")
            return
        changed = []
        for s in range(4):
            for a in range(3):
                if bets[s][a] != _N1_START_BETS[s][a]:
                    changed.append([s, a])
        if len(changed) == 0:
            print("⏳ my_bets is still page 3's table: set the bet of HEALTHY and REPLACE to 5 in the cell of warm-up "
                  "4, run it, then this check again.")
            return
        if changed == [[HEALTHY, REPLACE]] and bets[HEALTHY][REPLACE] == 5:
            print("✅ The bet of HEALTHY and REPLACE is 5: HEALTHY's chance of REPLACE went from %.2f to %.2f." % (
                _n1_softmax(_N1_START_BETS[HEALTHY])[REPLACE], _n1_softmax(bets[HEALTHY])[REPLACE]))
            return
        if len(changed) > 1:
            print("❌ %d bets of my_bets differ from page 3's; warm-up 4 changes one." % len(changed))
            print("   💡 Hint: start again from my_bets = start_bets(), then change only my_bets[HEALTHY][REPLACE].")
            return
        s, a = changed[0]
        if [s, a] != [HEALTHY, REPLACE]:
            print("❌ You changed the bet of %s and %s; warm-up 4 changes the bet of HEALTHY and REPLACE." % (
                STATES[s], ACTIONS[a]))
            print("   💡 Hint: the row comes first, then the column: my_bets[HEALTHY][REPLACE] = 5.")
            return
        print("❌ The bet of HEALTHY and REPLACE is %s; warm-up 4 sets it to 5." % _n1_short(bets[s][a]))
        print("   💡 Hint: my_bets[HEALTHY][REPLACE] = 5.")
    except Exception as error:
        print("❌ The check could not run (%s). Run the folded cells at the top again, then this cell." % error)


def throw_balls(chances, throws):
    """Throws the ball throws times on the roulette of a row of chances, with your make_roulette and where_it_lands,
    and prints how often each action came."""
    try:
        if not _n1_chances_ready(chances, "throw_balls"):
            return
        if not _n1_ready("make_roulette", "this cell") or not _n1_ready("where_it_lands", "this cell"):
            return
        ends = make_roulette(chances)
        counts = [0, 0, 0]
        for k in range(throws):
            slot = where_it_lands(ends, spin())
            counts[slot] = counts[slot] + 1
        print("In %s throws:  RUN %d times, SERVICE %d times, REPLACE %d times" % (
            format(throws, ","), counts[0], counts[1], counts[2]))
        print("Their shares:     RUN %.2f, SERVICE %.2f, REPLACE %.2f" % (
            counts[0] / throws, counts[1] / throws, counts[2] / throws))
        print("The chances:      RUN %.2f, SERVICE %.2f, REPLACE %.2f" % (chances[0], chances[1], chances[2]))
    except Exception as error:
        print("❌ throw_balls could not run (%s). Run the folded cells at the top again, then this cell." % error)


def show_picks(policy, picks):
    """Your pick_action, picks times in each state of a probability table: how often each action came, and the
    shares."""
    try:
        if not _n1_policy_ready(policy, "show_picks"):
            return
        if not _n1_ready("pick_action", "this cell"):
            return
        counts = []
        shares = []
        for s in range(4):
            counts.append([0, 0, 0])
            for k in range(picks):
                action = pick_action(policy, s)
                counts[s][action] = counts[s][action] + 1
            shares.append([counts[s][0] / picks, counts[s][1] / picks, counts[s][2] / picks])
        print("Your pick_action, %s times in each state: how often each action came" % format(picks, ","))
        show_counts(counts)
        print()
        print("Their shares, to set beside the probability table")
        show_chances(shares)
    except Exception as error:
        print("❌ show_picks could not run (%s). Run the folded cells at the top again, then this cell." % error)


def count_picks(how_many):
    """Your pick_bet, how_many times: how often each bet came."""
    try:
        if not _n1_ready("pick_bet", "this cell"):
            return
        counts = [[0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0]]
        for k in range(how_many):
            state, action = pick_bet()
            counts[state][action] = counts[state][action] + 1
        print("Your pick_bet, %s times: how often each bet came" % format(how_many, ","))
        show_counts(counts)
    except Exception as error:
        print("❌ count_picks could not run (%s). Run the folded cells at the top again, then this cell." % error)


def show_run(policy, weeks):
    """Plays one run with your pick_action and the kit's transition, and prints it week by week."""
    try:
        if not _n1_policy_ready(policy, "show_run"):
            return
        if not _n1_ready("pick_action", "this cell"):
            return
        state = HEALTHY
        print("week  state     action     money")
        for week in range(weeks):
            action = pick_action(policy, state)
            next_state, money = transition(state, action)
            print("%4d  %-8s  %-8s  %6s" % (week + 1, STATES[state], ACTIONS[action], _kit_dollars(money)))
            state = next_state
    except Exception as error:
        print("❌ show_run could not run (%s). Run the folded cells at the top again, then this cell." % error)

print("✅ The checks of part 3 are ready.")

## 1. The forward pass, given

The folded cell above holds the forward pass. This notebook calls three of its functions:

- `start_bets()`: a fresh copy of page 3's table of bets, picked at random between -10 and 10.
- `probability_table(bets)`: softmax on every row of a table of bets, so a larger bet gets a larger chance.
- `average_G1(bets)`: the average $G_1$ of `RUNS` runs of `WEEKS` weeks with the policy of a table of bets, in
  cents. Every call plays the same runs, so two tables of bets are compared on the same luck.

In [ ]:
# Page 3's table of bets, its probability table and its average G1
bets = start_bets()
show_bets(bets)
print()
show_chances(probability_table(bets))
print()
show_money("The average G1 of page 3's bets:", play(average_G1, bets))

**Reading the result.** -\$172.24: the first point of page 3's reward plot. The backward pass raises it one bet at a
time.

## 2. A bet at random

Each update of proto-REINFORCE picks one bet at random, every bet as likely as any other: a row, the state, and
a column, the action. Python's `random.randint(0, 3)` gives 0, 1, 2 or 3, each as likely as the others: a row.
`random.randint(0, 2)` gives a column. Run the next cell a few times.

In [ ]:
# A row at random, then a column at random
state = random.randint(0, 3)
action = random.randint(0, 2)
print("The bet picked at random:", STATES[state], "and", ACTIONS[action])

### 🎯 Task A. A bet at random

Write `pick_bet()`: a row at random, then a column at random, given back together. It takes no inputs.

In [ ]:
# 🎯 YOUR TURN, task A: a bet at random
def pick_bet():
    # Gives a bet picked at random, every bet as likely as any other: its row, the state, and its column, the action.
    state = ...                         # 🎯 YOUR TURN: a row at random, one of the 4 states
    action = ...                        # 🎯 YOUR TURN: a column at random, one of the 3 actions
    return state, action

In [ ]:
#@title 💡 Hint 1 for task A (double-click to open) { display-mode: "form" }
# The two lines of the cell above: random.randint(0, 3) for one of the 4 rows, random.randint(0, 2) for one of
# the 3 columns.

In [ ]:
#@title 💡 Hint 2 for task A (double-click to open) { display-mode: "form" }
#     state = random.randint(0, 3)
#     action = random.randint(0, 2)

In [ ]:
# Check task A
check_pick_bet(pick_bet)

In [ ]:
# Your pick_bet, 1,200 times
count_picks(1200)

**Reading the result.** Each of the 12 bets comes about 100 times in 1,200 picks: every bet is as likely as any
other.

## 3. Before and After

Page 3's update 1 picked the bet of HEALTHY and RUN, 0.4. proto-REINFORCE asks how the average $G_1$ changes
when that bet rises a little, by a hop of 0.01. Before is the average $G_1$ of the bets as they are. After is the
average $G_1$ with that one bet a hop higher, on the same luck. The next cell works it out.

In [ ]:
# Update 1 of page 3, by hand
hop = 0.01                                      # the hop: a small change of one bet
bets = start_bets()                             # page 3's table of bets
state, action = HEALTHY, RUN                    # the bet that page 3's update 1 picked at random
hopped = copy_bets(bets)                        # a copy: the table itself stays as it is
hopped[state][action] = hopped[state][action] + hop        # one hop higher

before = play(average_G1, bets)                 # Before: the bets as they are
after = play(average_G1, hopped)                # After: one bet a hop higher, on the same luck
show_money("Before:", before)
show_money("After: ", after)

**Reading the result.** The hop raised the average $G_1$ from -\$172.24 to -\$170.50, by \$1.74. The slope, $\nabla$ (nabla), is
the change divided by the hop, rounded to a whole number as on page 3:

$$\nabla = \frac{\text{After} - \text{Before}}{\text{hop}} = \frac{1.74}{0.01} = 174$$

A positive slope says: raise this bet. A negative slope says: lower it.

### 🎯 Task B. The slope

Write `slope(bets, state, action, hop)`: Before, After with the bet a hop higher in a copy of the bets, and the
change divided by the hop.

In [ ]:
# 🎯 YOUR TURN, task B: the slope, nabla
def slope(bets, state, action, hop):
    # How fast the average G1 rises when the bet of state and action rises, measured with a hop.
    # Gives nabla, a whole number, as on page 3.
    before = average_G1(bets)                   # Before: the bets as they are
    hopped = copy_bets(bets)                    # a copy: the table itself stays as it is
    hopped[state][action] = ...                 # 🎯 YOUR TURN: the bet, one hop higher
    after = ...                                 # 🎯 YOUR TURN: After, the average G1 of the hopped bets
    nabla = ...                                 # 🎯 YOUR TURN: the change, after - before, divided by the hop
    return round(nabla)

In [ ]:
#@title 💡 Hint 1 for task B (double-click to open) { display-mode: "form" }
# The cell above, inside a function: one hop added to the bet of the copy, After from average_G1 of the copy,
# then the change divided by the hop.

In [ ]:
#@title 💡 Hint 2 for task B (double-click to open) { display-mode: "form" }
#     hopped[state][action] = hopped[state][action] + hop
#     after = average_G1(hopped)
#     nabla = (after - before) / hop

In [ ]:
# Check task B
check_slope(slope)

In [ ]:
# The slope of page 3's update 1
nabla = play(slope, bets, HEALTHY, RUN, hop)
show_value("nabla =", nabla)

**Reading the result.** nabla = 174: near this bet, the average $G_1$ rises by \$174 for each unit of bet.

## 4. The update

A positive slope says: raise this bet. The update moves the bet by $\alpha$ times the slope:

$$w = w + \alpha \nabla$$

In weekend 2 a loss had to go down: $w - \alpha \nabla$. Here the average $G_1$ has to go up:
$w + \alpha \nabla$. $\alpha = 0.01$ keeps each move small. Page 3's update 1:
$w = 0.40 + 0.01 \times 174 = 2.14$.

### 🎯 Task C. The update of one bet

Write `update_bet(bets, state, action, nabla, alpha)`: the new bet, $w + \alpha \nabla$. The given lines put it
into the table, in cents, and give it back.

In [ ]:
# 🎯 YOUR TURN, task C: the update of one bet
def update_bet(bets, state, action, nabla, alpha):
    # The update of one bet, w = w + alpha nabla, in cents. Changes the table of bets and gives the new bet.
    w = bets[state][action]                     # the bet as it is
    new_bet = ...                               # 🎯 YOUR TURN: w plus alpha times nabla
    bets[state][action] = round(new_bet, 2)     # into the table, in cents, as on page 3
    return bets[state][action]

In [ ]:
#@title 💡 Hint 1 for task C (double-click to open) { display-mode: "form" }
# w is the bet as it is, alpha the step and nabla the slope: the new bet is w plus alpha times nabla.

In [ ]:
#@title 💡 Hint 2 for task C (double-click to open) { display-mode: "form" }
#     new_bet = w + alpha * nabla

In [ ]:
# Check task C
check_update_bet(update_bet)

💭 **Think first.** With 2.14 in place of 0.40, softmax gives HEALTHY new chances. Will the average $G_1$ go up
or down?

In [ ]:
# Page 3's update 1: the new bet, the new chances, and the average G1 again, on the same luck
alpha = 0.01                                    # alpha: keeps each move small
bets = start_bets()                             # page 3's table of bets
new_bet = play(update_bet, bets, HEALTHY, RUN, nabla, alpha)
if new_bet is not None:                         # once update_bet works
    show_value("The new bet of HEALTHY and RUN:", new_bet)
    print()
    show_chances(probability_table(bets))
    print()
    show_money("The average G1 after update 1:", play(average_G1, bets))

**Reading the result.** Up: from -\$172.24 to \$73.57, the second point of page 3's reward plot. HEALTHY's chance of RUN
went from 0.47 to 0.84: one bet moved, and on average the van now makes money.

## 5. proto-REINFORCE

Now the whole loop of page 3. Each update plays new runs, picks a bet at random, measures its slope, updates the
bet, and adds one point to the reward plot: the average $G_1$ after the update.

### 🎯 Task D. The loop

Write `proto_reinforce(bets, updates, hop, alpha)`. Its first point and the first line of each update are given:
`new_luck()` deals the update new runs. Each line you write calls one of your functions.

In [ ]:
# 🎯 YOUR TURN, task D: proto-REINFORCE
def proto_reinforce(bets, updates, hop, alpha):
    # bets: the table of bets, changed by every update. updates: how many updates.
    # Gives the points of the reward plot: the average G1 before the first update, then after each update.
    points = [average_G1(bets)]                 # the first point: the bets as they are
    for update in range(updates):
        new_luck()                              # new runs for this update
        state, action = ...                     # 🎯 YOUR TURN: a bet at random, with your pick_bet
        nabla = ...                             # 🎯 YOUR TURN: the slope of that bet, with your slope
        ...                                     # 🎯 YOUR TURN: the update of that bet, with your update_bet
        points.append(...)                      # 🎯 YOUR TURN: the average G1 after the update
    return points

In [ ]:
#@title 💡 Hint 1 for task D (double-click to open) { display-mode: "form" }
# Your functions, in the order of an update: pick_bet for the bet, slope for nabla, update_bet for the update,
# average_G1 for the point.
# update_bet changes the table itself, so its line is the call alone.

In [ ]:
#@title 💡 Hint 2 for task D (double-click to open) { display-mode: "form" }
#         state, action = pick_bet()
#         nabla = slope(bets, state, action, hop)
#         update_bet(bets, state, action, nabla, alpha)
#         points.append(average_G1(bets))

In [ ]:
# Check task D
check_proto_reinforce(proto_reinforce)

### 300 updates

💭 **Think first.** Which action do you expect to be the likeliest in each state after 300 updates?

The next cell plays 300 updates from page 3's bets, each with three averages of 1,000 runs: under a minute on Colab.

In [ ]:
# 300 updates of proto-REINFORCE, from page 3's bets (under a minute on Colab)
bets = start_bets()
points = play(proto_reinforce, bets, 300, hop, alpha)
plot_rewards(points)

**Reading the result.** The first point is page 3's start, -\$172.24. Update 1 picked HEALTHY and RUN, the bet of page
3's update 1, on new runs: nabla = 224, the bet went from 0.40 to 2.64, and the average to \$123.82. After update 300 the
average is \$236.16. The row of dashes is the best strategy's average $G_1$ on the runs of update 300, \$244.03. In 167 of the
300 updates nabla was 0: the hop of 0.01 changed no run. Every update that picked SHAKY's bet on REPLACE, 10, is one
of them: REPLACE is almost certain already, and that bet never moves.

In [ ]:
# The table of bets after the updates, and its probability table
if points is not None:                          # once the loop above has run
    show_bets(bets)
    print()
    show_chances(probability_table(bets))

### The best strategy

After 300 updates the likeliest action in each state is RUN, SERVICE, REPLACE, REPLACE: the best strategy. Of the 81 strategies that take one
fixed action in each state, it has the highest average $G_1$ over a quarter, \$242.08, computed exactly from the van's
chances; on the runs of update 300 it gets \$244.03, the line of the plot. The table still takes another action now and
then: a FAILING van runs in 12 weeks of 100 and goes to the garage in 31. So the last point, \$236.16, sits a little
below the line.

## What you built

- **The forward pass**, given: `average_G1` gives the average $G_1$ of 1,000 runs of the policy of a table of bets.
- **A bet at random**: `pick_bet` picks a row and a column.
- **The backward pass**: `slope` raises one bet by a hop of 0.01, plays the same runs again and gives nabla.
- **The update**: `update_bet` moves the bet, $w = w + \alpha \nabla$.
- **The loop**: `proto_reinforce` repeats it, one bet picked at random each time, and gives the points of the
  reward plot.

Next, notebook 2 runs the same algorithm on another game of the arcade, Last-Minute Pricing:
[proto-REINFORCE, notebook 2: Last-Minute Pricing](https://colab.research.google.com/github/eth-ainit-hs26/w3-cx-public/blob/main/03_cx_proto_reinforce_2_last_minute_pricing_student.ipynb).